## Windows: HADOOP_HOME 필수
이 노트북 실행 전에 **반드시** 아래 배치 파일로 Jupyter를 시작하세요:
```
start_09_spark_gcs.bat
```
(또는 `start_jupyter_with_hadoop.bat`)  
Anaconda/Cursor에서 직접 열면 HADOOP_HOME 오류가 발생합니다.

In [1]:
# Windows: 반드시 첫 셀에서 실행. SparkContext 생성 전에 HADOOP_HOME 설정
import os
_hadoop_dirs = [
    r"E:\IT_SPACES\AI\ZoomCamp\DE\06\tools\hadoop-3.3.5",
    r"E:\IT_SPACES\AI\ZoomCamp\DE\tools\hadoop-3.3.5",
]
for _d in _hadoop_dirs:
    if os.path.isdir(_d):
        os.environ["HADOOP_HOME"] = _d
        os.environ["PATH"] = os.environ.get("PATH", "") + os.pathsep + os.path.join(_d, "bin")
        print("HADOOP_HOME:", _d)
        break
else:
    print("경고: hadoop 폴더 없음. start_jupyter_with_hadoop.bat 또는 install_winutils.py 실행 필요.")

HADOOP_HOME: E:\IT_SPACES\AI\ZoomCamp\DE\06\tools\hadoop-3.3.5


In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
from pyspark.context import SparkContext

In [3]:
import os

# Windows: hadoop.home.dir 필수 - SparkConf에 직접 설정 (JVM에 전달됨)
HADOOP_DIR = r"E:\IT_SPACES\AI\ZoomCamp\DE\06\tools\hadoop-3.3.5"
if not os.path.isdir(HADOOP_DIR):
    HADOOP_DIR = r"E:\IT_SPACES\AI\ZoomCamp\DE\tools\hadoop-3.3.5"
os.environ["HADOOP_HOME"] = HADOOP_DIR if os.path.isdir(HADOOP_DIR) else ""
if os.environ["HADOOP_HOME"]:
    os.environ["PATH"] = os.environ.get("PATH", "") + os.pathsep + os.path.join(HADOOP_DIR, "bin")

# credentials: 아래에 경로 지정 또는 자동 탐색
credentials_location = None  # 예: r"C:\Users\사용자\.google\credentials\google_credentials.json"
if not credentials_location or not os.path.isfile(credentials_location):
    _candidates = [
        os.path.join(os.path.expanduser("~"), ".google", "credentials", "google_credentials.json"),
        os.path.join(os.getcwd(), "google_credentials.json"),
        os.path.join(r"E:\IT_SPACES\AI\ZoomCamp\DE\06\Batch\code", "google_credentials.json"),
    ]
    credentials_location = next((p for p in _candidates if os.path.isfile(p)), None)
if not credentials_location or not os.path.isfile(credentials_location):
    raise FileNotFoundError(
        "google_credentials.json 없음. GCP 서비스 계정 키를 아래 중 한 곳에 두세요:\n"
        "  - C:\\Users\\<사용자>\\.google\\credentials\\google_credentials.json\n"
        "  - E:\\IT_SPACES\\AI\\ZoomCamp\\DE\\06\\Batch\\code\\google_credentials.json"
    )
credentials_location = os.path.normpath(credentials_location).replace("\\", "/")  # Spark/Hadoop은 / 선호
print("credentials:", credentials_location)

conf = SparkConf() \
    .setMaster('local[*]') \
    .setAppName('test') \
    .set("spark.jars", "./lib/gcs-connector-hadoop3-2.2.5.jar") \
    .set("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
    .set("spark.hadoop.google.cloud.auth.service.account.json.keyfile", credentials_location)
if os.path.isdir(HADOOP_DIR):
    conf = conf.set("spark.hadoop.hadoop.home.dir", HADOOP_DIR)

credentials: E:/IT_SPACES/AI/ZoomCamp/DE/06/Batch/code/google_credentials.json


In [4]:
sc = SparkContext(conf=conf)

hadoop_conf = sc._jsc.hadoopConfiguration()

hadoop_conf.set("fs.AbstractFileSystem.gs.impl",  "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
hadoop_conf.set("fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
hadoop_conf.set("fs.gs.auth.service.account.json.keyfile", credentials_location)
hadoop_conf.set("fs.gs.auth.service.account.enable", "true")

In [5]:
spark = SparkSession.builder \
    .config(conf=sc.getConf()) \
    .getOrCreate()

In [6]:
df_green = spark.read.parquet('gs://de-zoomcamp-zakard-2026/pq/green/*/*')

In [7]:
df_green.count()

1810569

In [8]:
df_green.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- lpep_pickup_datetime: timestamp (nullable = true)
 |-- lpep_dropoff_datetime: timestamp (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- trip_type: integer (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [10]:
# 컬럼명의 대소문자를 스키마에 맞게 수정했습니다.
df_green.select('VendorID', 'lpep_pickup_datetime', 'trip_distance').show(10)

+--------+--------------------+-------------+
|VendorID|lpep_pickup_datetime|trip_distance|
+--------+--------------------+-------------+
|       2| 2020-01-18 03:52:15|         0.69|
|       2| 2020-01-06 08:36:04|         3.39|
|       2| 2020-01-06 04:57:35|         0.91|
|       1| 2020-01-28 13:51:34|          2.8|
|       2| 2020-01-03 18:24:16|         7.09|
|       1| 2020-01-25 23:43:20|          3.1|
|       2| 2020-01-06 19:20:41|         1.62|
|       1| 2020-01-04 12:35:31|          1.3|
|       2| 2020-01-11 09:51:16|         1.66|
|       1| 2020-01-14 15:18:18|          0.0|
+--------+--------------------+-------------+
only showing top 10 rows



In [12]:
from pyspark.sql import functions as F

# 'vendor_id' -> 'VendorID'로 변경
df_green.groupBy('VendorID').count().show()

+--------+-------+
|VendorID|  count|
+--------+-------+
|    NULL| 564139|
|       1| 212352|
|       2|1034078|
+--------+-------+

